In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
def repeat_kv(x, n_rep):
    if n_rep == 1:
        return x
    bs, slen, n_kv_heads, head_dim = x.shape
    x = x[:, :, :, None, :].expand(bs, slen, n_kv_heads, n_rep, head_dim)
    return x.reshape(bs, slen, n_kv_heads * n_rep, head_dim)

class Attention(nn.Module):
    def __init__ (self, config):
        super().__init__()
        self.n_heads = config.n_heads
        self.n_kv_heads = config.n_kv_heads
        self.head_dim = config.dim // config.n_heads
        self.n_rep = config.n_heads // config.n_kv_heads

        self.wq = nn.Linear(config.dim, self.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(config.dim, self.n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(config.dim, self.n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(self.n_heads * self.head_dim, config.dim, bias=False)
    
    def forward(self, x, freqs_cis, mask=None):
        bsz, seqlen, hidden_dim = x.shape
        
        xq = self.wq(x).view(bsz, seqlen, self.n_heads, self.head_dim)
        xk = self.wk(x).view(bsz, seqlen, self.n_kv_heads, self.head_dim)
        xv = self.wv(x).view(bsz, seqlen, self.n_kv_heads, self.head_dim)

        xq, xk = apply_rotary_emb(xq, xk, freqs_cis)

        xk = repeat_kv(xk, self.n_rep)
        xv = repeat_kv(xv, self.n_rep)
        xq, xk, xv = [t.transpose(1, 2) for t in (xq, xk, xv)]

        scores = torch.matmul(xq, xk.transpose(2, 3)) / (self.head_dim ** 0.5)
        if mask is not None:
            scores = scores + mask
            scores = F.softmax(scores.float(), dim=-1).type_as(scores)
            output = torch.matmul(scores, xv)
            output = output.transpose(1,2).contiguous().view(bsz, seqlen, -1)
            return self.wo(output)
